# Higher-Order Relations (Modal GPU)

This notebook is designed to run on Modal's GPU infrastructure.

**To run:**
```bash
# Start Jupyter server on Modal GPU
modal run modal_app.py

# Then open the provided URL and run this notebook
```

In [ ]:
import sys
sys.path.insert(0, '/root')  # Modal container path

import jax
import jax.numpy as jnp
from jax import vmap
import numpy as np
import matplotlib.pyplot as plt

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

# Verify GPU
assert any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in jax.devices()), \
    "No GPU detected! Make sure you're running on Modal with GPU."

In [ ]:
from models import (
    get_eff, pw_contact_map, con_auc,
    plot_contact_map, plot_contact_comparison, plot_training_curves,
    # MRF
    train_mrf, mrf_forward, mrf_fields, mrf_jacobian, mrf_hessian,
    # LAE
    train_lae, lae_forward, lae_fields, lae_jacobian, lae_hessian,
    # VAE
    train_vae, vae_forward, vae_fields, vae_jacobian, vae_hessian,
)

print("Models loaded successfully")

In [ ]:
# =============================================================================
# OPTION C: P0AA25 (L=101) — Run this cell OR one of the previous ones
# =============================================================================
# Use for: Jacobian + Hessian (33 GB, fits on H200)
# Protein: AF-P0AA25-F1 (UniProt P0AA25)

msa_path = "/data/AF-P0AA25-F1-msa_v6.npz"
contacts_path = "/data/AF-P0AA25-F1-model_v6_contacts.npz"

msa_data = np.load(msa_path)
contacts_data = np.load(contacts_path)

# X is already one-hot encoded (N, L, A)
X = msa_data["X"]
W = msa_data["weights"]  # Pre-computed sequence weights
cons = contacts_data["contacts"]

# Position mapping to original alignment (with gaps)
non_gap = msa_data["non_gap"]

N, L, A = X.shape
d = L * A

print(f"Loaded: {msa_path}")
print(f"MSA: {N} sequences, L={L} positions, A={A} amino acids, d={d}")
print(f"Effective sequences: {W.sum():.1f}")
print(f"Non-gap positions: {len(non_gap)} (from original {msa_data['X_raw'].shape[1]})")
print(f"\nHessian would be: {d**3 * 4 / 1e9:.1f} GB (f32)")

In [ ]:
# Compute sequence weights if not pre-computed (Option C has them)
if 'W' not in dir() or W is None:
    X_raw = data["X"]  # Integer-encoded for weight computation
    W = get_eff(X_raw)
    print(f"Computed sequence weights: {W.sum():.1f} effective sequences")
else:
    print(f"Using pre-computed weights: {W.sum():.1f} effective sequences")

X_jax, W_jax = jnp.array(X), jnp.array(W)

---
# Markov Random Field (MRF)

GPU-accelerated training.

In [ ]:
%%time
print("Training MRF1...")
params_mrf1, losses_mrf1 = train_mrf(X_jax, W_jax, lam=0.01, use_bias=True, n_epochs=200)

In [ ]:
# 0th Order: Fields (sitewise conservation)
fields_mrf1 = mrf_fields(params_mrf1, L, A)
print(f"MRF1 fields shape: {fields_mrf1.shape}")
print(f"Field strength range: [{float(fields_mrf1.min()):.3f}, {float(fields_mrf1.max()):.3f}]")

In [ ]:
# 1st Order: Jacobian (pairwise coevolution)
jac_mrf1 = mrf_jacobian(params_mrf1, L, A)
cons_pred_mrf1 = pw_contact_map(jac_mrf1)
auc_mrf1 = con_auc(cons_pred_mrf1, cons)

print(f"MRF1: AUC={auc_mrf1:.4f}")

In [ ]:
%%time
print("Training MRF2...")
params_mrf2, losses_mrf2 = train_mrf(X_jax, W_jax, lam=0.01, use_bias=True, n_epochs=200)

In [ ]:
# 0th Order: Fields (sitewise conservation)
fields_mrf2 = mrf_fields(params_mrf2, L, A)
print(f"MRF2 fields shape: {fields_mrf2.shape}")
print(f"Field strength range: [{float(fields_mrf2.min()):.3f}, {float(fields_mrf2.max()):.3f}]")

In [ ]:
# 1st Order: Jacobian (pairwise coevolution)
jac_mrf2 = mrf_jacobian(params_mrf2, L, A)
cons_pred_mrf2 = pw_contact_map(jac_mrf2)
auc_mrf2 = con_auc(cons_pred_mrf2, cons)

print(f"MRF2: AUC={auc_mrf2:.4f}")

---
# Linear Autoencoder (LAE)

In [ ]:
%%time
print("Training LAE1...")
params_lae1, losses_lae1 = train_lae(X_jax, W_jax, rank=512, lam_w=0.01, use_bias=True, n_epochs=135)

In [ ]:
# 0th Order: Fields (sitewise conservation)
fields_lae1 = lae_fields(params_lae1, L, A)
print(f"LAE1 fields shape: {fields_lae1.shape}")
print(f"Field strength range: [{float(fields_lae1.min()):.3f}, {float(fields_lae1.max()):.3f}]")

In [ ]:
# 1st Order: Jacobian (pairwise coevolution)
jac_lae1 = lae_jacobian(params_lae1, L, A)
cons_pred_lae1 = pw_contact_map(jac_lae1)
auc_lae1 = con_auc(cons_pred_lae1, cons)

print(f"LAE1: AUC={auc_lae1:.4f}")

In [ ]:
%%time
print("Training LAE2...")
params_lae2, losses_lae2 = train_lae(X_jax, W_jax, rank=512, lam_w=0.01, use_bias=True, n_epochs=135)

In [ ]:
# 0th Order: Fields (sitewise conservation)
fields_lae2 = lae_fields(params_lae2, L, A)
print(f"LAE2 fields shape: {fields_lae2.shape}")
print(f"Field strength range: [{float(fields_lae2.min()):.3f}, {float(fields_lae2.max()):.3f}]")

In [ ]:
# 1st Order: Jacobian (pairwise coevolution)
jac_lae2 = lae_jacobian(params_lae2, L, A)
cons_pred_lae2 = pw_contact_map(jac_lae2)
auc_lae2 = con_auc(cons_pred_lae2, cons)

print(f"LAE2: AUC={auc_lae2:.4f}")

---
# Variational Autoencoder (VAE)

In [ ]:
%%time
print("Training VAE1...")
params_vae1, config_vae1, losses_vae1 = train_vae(
    X_jax, W_jax, 
    enc_dims=[512, 512], rank=32, dec_dims=[512, 512],
    n_epochs=400
)

In [ ]:
# Extract config for VAE1
n_enc1 = len(config_vae1["enc_dims"])
n_dec1 = len(config_vae1["dec_dims"])
use_blosum1 = config_vae1["use_blosum"]

# 0th Order: Fields (sitewise conservation)
fields_vae1 = vae_fields(params_vae1, L, A, n_enc1, n_dec1, use_blosum1)
print(f"VAE1 fields shape: {fields_vae1.shape}")
print(f"Field strength range: [{float(fields_vae1.min()):.3f}, {float(fields_vae1.max()):.3f}]")

In [ ]:
# 1st Order: Jacobian (pairwise coevolution)
jac_vae1 = vae_jacobian(params_vae1, L, A, n_enc1, n_dec1, use_blosum1)
cons_pred_vae1 = pw_contact_map(jac_vae1)
auc_vae1 = con_auc(cons_pred_vae1, cons)

print(f"VAE1: AUC={auc_vae1:.4f}")

In [ ]:
%%time
print("Training VAE2...")
params_vae2, config_vae2, losses_vae2 = train_vae(
    X_jax, W_jax,
    enc_dims=[1024], rank=256, dec_dims=[1024],
    use_blosum=False,
    n_epochs=400
)

In [ ]:
# Extract config for VAE2
n_enc2 = len(config_vae2["enc_dims"])
n_dec2 = len(config_vae2["dec_dims"])
use_blosum2 = config_vae2["use_blosum"]

# 0th Order: Fields (sitewise conservation)
fields_vae2 = vae_fields(params_vae2, L, A, n_enc2, n_dec2, use_blosum2)
print(f"VAE2 fields shape: {fields_vae2.shape}")
print(f"Field strength range: [{float(fields_vae2.min()):.3f}, {float(fields_vae2.max()):.3f}]")

In [ ]:
# 1st Order: Jacobian (pairwise coevolution)
jac_vae2 = vae_jacobian(params_vae2, L, A, n_enc2, n_dec2, use_blosum2)
cons_pred_vae2 = pw_contact_map(jac_vae2)
auc_vae2 = con_auc(cons_pred_vae2, cons)

print(f"VAE2: AUC={auc_vae2:.4f}")

---
# Summary

In [ ]:
print("\n" + "=" * 50)
print("RESULTS COMPARISON (Modal GPU)")
print("=" * 50)
print(f"{'Model':<12} {'Contact AUC':<15}")
print("-" * 50)
print(f"{'MRF1':<12} {auc_mrf1:<15.4f}")
print(f"{'MRF2':<12} {auc_mrf2:<15.4f}")
print("-" * 50)
print(f"{'LAE1':<12} {auc_lae1:<15.4f}")
print(f"{'LAE2':<12} {auc_lae2:<15.4f}")
print("-" * 50)
print(f"{'VAE1':<12} {auc_vae1:<15.4f}")
print(f"{'VAE2':<12} {auc_vae2:<15.4f}")
print("=" * 50)

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses_mrf1, label="MRF1")
axes[0].plot(losses_mrf2, label="MRF2")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("MRF Training")
axes[0].legend()

axes[1].plot(losses_lae1, label="LAE1")
axes[1].plot(losses_lae2, label="LAE2")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("LAE Training")
axes[1].legend()

axes[2].plot(losses_vae1, label="VAE1")
axes[2].plot(losses_vae2, label="VAE2")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].set_title("VAE Training")
axes[2].legend()

plt.tight_layout()
plt.show()

## Contact Maps

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

axes[0, 0].imshow(cons, cmap="gray_r")
axes[0, 0].set_title("True Contacts")

axes[0, 1].imshow(np.array(cons_pred_mrf1), cmap="viridis")
axes[0, 1].set_title(f"MRF1 (AUC={auc_mrf1:.3f})")
axes[0, 2].imshow(np.array(cons_pred_mrf2), cmap="viridis")
axes[0, 2].set_title(f"MRF2 (AUC={auc_mrf2:.3f})")

axes[1, 0].axis("off")
axes[1, 1].imshow(np.array(cons_pred_lae1), cmap="viridis")
axes[1, 1].set_title(f"LAE1 (AUC={auc_lae1:.3f})")
axes[1, 2].imshow(np.array(cons_pred_lae2), cmap="viridis")
axes[1, 2].set_title(f"LAE2 (AUC={auc_lae2:.3f})")

axes[2, 0].axis("off")
axes[2, 1].imshow(np.array(cons_pred_vae1), cmap="viridis")
axes[2, 1].set_title(f"VAE1 (AUC={auc_vae1:.3f})")
axes[2, 2].imshow(np.array(cons_pred_vae2), cmap="viridis")
axes[2, 2].set_title(f"VAE2 (AUC={auc_vae2:.3f})")

for ax in axes.flat:
    if ax.has_data():
        ax.set_xlabel("Position")
        ax.set_ylabel("Position")

plt.tight_layout()
plt.show()

## Fields Tensors (0th Order)

Fields represent sitewise conservation — the log-probability at the zero reference point.
Not a derivative, but the constant term in the Taylor expansion.

In [ ]:
print("Fields tensor shapes:")
print(f"  MRF1: {fields_mrf1.shape}")
print(f"  MRF2: {fields_mrf2.shape}")
print(f"  LAE1: {fields_lae1.shape}")
print(f"  LAE2: {fields_lae2.shape}")
print(f"  VAE1: {fields_vae1.shape}")
print(f"  VAE2: {fields_vae2.shape}")
print(f"\nAll are (L, A) = ({L}, {A})")
print(f"Total elements per tensor: {L * A:,}")

## Jacobian Tensors (1st Order)

In [ ]:
print("Jacobian tensor shapes:")
print(f"  MRF1: {jac_mrf1.shape}")
print(f"  MRF2: {jac_mrf2.shape}")
print(f"  LAE1: {jac_lae1.shape}")
print(f"  LAE2: {jac_lae2.shape}")
print(f"  VAE1: {jac_vae1.shape}")
print(f"  VAE2: {jac_vae2.shape}")
print(f"\nAll are (L, A, L, A) = ({L}, {A}, {L}, {A})")
print(f"Total elements per tensor: {L * A * L * A:,}")

---
## Save Fields

Save field tensors to Modal volume as .npy files.

In [ ]:
# Save Fields as .npy files
np.save('/data/fields_mrf1.npy', np.array(fields_mrf1))
np.save('/data/fields_mrf2.npy', np.array(fields_mrf2))
np.save('/data/fields_lae1.npy', np.array(fields_lae1))
np.save('/data/fields_lae2.npy', np.array(fields_lae2))
np.save('/data/fields_vae1.npy', np.array(fields_vae1))
np.save('/data/fields_vae2.npy', np.array(fields_vae2))

print("Fields saved to /data/:")
print("  fields_mrf1.npy, fields_mrf2.npy")
print("  fields_lae1.npy, fields_lae2.npy")
print("  fields_vae1.npy, fields_vae2.npy")

---
# Save Jacobians

Save Jacobian tensors to Modal volume as .npy files.

In [ ]:
# Save Jacobians as .npy files
np.save('/data/jac_mrf1.npy', np.array(jac_mrf1))
np.save('/data/jac_mrf2.npy', np.array(jac_mrf2))
np.save('/data/jac_lae1.npy', np.array(jac_lae1))
np.save('/data/jac_lae2.npy', np.array(jac_lae2))
np.save('/data/jac_vae1.npy', np.array(jac_vae1))
np.save('/data/jac_vae2.npy', np.array(jac_vae2))

print("Jacobians saved to /data/:")
print("  jac_mrf1.npy, jac_mrf2.npy")
print("  jac_lae1.npy, jac_lae2.npy")
print("  jac_vae1.npy, jac_vae2.npy")

---
# Hessian (2nd Order) Computation

Extend Jacobian (1st order) to Hessian (2nd order) to capture epistatic curvature.

| Order | Tensor | Shape | Meaning | L=60 Memory |
|-------|--------|-------|---------|-------------|
| 0th | Fields | (L, A) | log p(x=0) | 4.8 KB |
| 1st | Jacobian | (L, A, L, A) | ∂log p / ∂x | 5.8 MB |
| 2nd | Hessian | (L, A, L, A, L, A) | ∂²log p / ∂x² | 6.9 GB |

**Note:** Only run this section with the **subsection data** (Option B, L=60). Full MSA (L=252) would require 552 GB.

In [ ]:
# Verify Hessian will fit in GPU memory
# L=60 → 6.9 GB (T4/A100), L=101 → 33 GB (H200/B200), L=252 → 552 GB (too large)
hess_gb = (L * A) ** 3 * 4 / 1e9
if hess_gb > 100:
    print(f"WARNING: L={L}, Hessian={hess_gb:.1f} GB — too large for any GPU")
    print("Use subsectioned data (Option B) or wait for larger GPUs")
elif hess_gb > 50:
    print(f"L={L}, d={L*A}, Hessian: {hess_gb:.1f} GB — requires H200 (141 GB) or B200 (192 GB)")
elif hess_gb > 20:
    print(f"L={L}, d={L*A}, Hessian: {hess_gb:.1f} GB — requires A100-80GB/H100/H200")
else:
    print(f"L={L}, d={L*A}, Hessian: {hess_gb:.1f} GB — fits on T4+")

In [ ]:
%%time
print("Computing MRF1 Hessian...")
hess_mrf1 = mrf_hessian(params_mrf1, L, A, verbose=True)
print(f"Shape: {hess_mrf1.shape}, Memory: {hess_mrf1.nbytes / 1e9:.1f} GB")

In [ ]:
%%time
print("Computing MRF2 Hessian...")
hess_mrf2 = mrf_hessian(params_mrf2, L, A, verbose=True)
print(f"Shape: {hess_mrf2.shape}, Memory: {hess_mrf2.nbytes / 1e9:.1f} GB")

In [ ]:
%%time
print("Computing LAE1 Hessian...")
hess_lae1 = lae_hessian(params_lae1, L, A, verbose=True)
print(f"Shape: {hess_lae1.shape}, Memory: {hess_lae1.nbytes / 1e9:.1f} GB")

In [ ]:
%%time
print("Computing LAE2 Hessian...")
hess_lae2 = lae_hessian(params_lae2, L, A, verbose=True)
print(f"Shape: {hess_lae2.shape}, Memory: {hess_lae2.nbytes / 1e9:.1f} GB")

In [ ]:
%%time
print("Computing VAE1 Hessian...")
hess_vae1 = vae_hessian(params_vae1, L, A, n_enc1, n_dec1, use_blosum1, verbose=True)
print(f"Shape: {hess_vae1.shape}, Memory: {hess_vae1.nbytes / 1e9:.1f} GB")

In [ ]:
%%time
print("Computing VAE2 Hessian...")
hess_vae2 = vae_hessian(params_vae2, L, A, n_enc2, n_dec2, use_blosum2, verbose=True)
print(f"Shape: {hess_vae2.shape}, Memory: {hess_vae2.nbytes / 1e9:.1f} GB")

In [ ]:
print("Hessian tensor shapes:")
print(f"  MRF1: {hess_mrf1.shape}")
print(f"  MRF2: {hess_mrf2.shape}")
print(f"  LAE1: {hess_lae1.shape}")
print(f"  LAE2: {hess_lae2.shape}")
print(f"  VAE1: {hess_vae1.shape}")
print(f"  VAE2: {hess_vae2.shape}")
print(f"\nAll are (L, A, L, A, L, A) = ({L}, {A}, {L}, {A}, {L}, {A})")
print(f"Total elements per tensor: {L * A * L * A * L * A:,}")

## Save Hessians

Save Hessian tensors to Modal volume. Due to size (~7 GB each), we save as separate .npy files.

In [ ]:
# Save Hessians as .npy files
np.save('/data/hess_mrf1.npy', np.array(hess_mrf1))
np.save('/data/hess_mrf2.npy', np.array(hess_mrf2))
np.save('/data/hess_lae1.npy', np.array(hess_lae1))
np.save('/data/hess_lae2.npy', np.array(hess_lae2))
np.save('/data/hess_vae1.npy', np.array(hess_vae1))
np.save('/data/hess_vae2.npy', np.array(hess_vae2))

print("Hessians saved to /data/:")
print("  hess_mrf1.npy, hess_mrf2.npy")
print("  hess_lae1.npy, hess_lae2.npy")
print("  hess_vae1.npy, hess_vae2.npy")